In [ ]:
# Notebook 08: Supervised Fine-Tuning (SFT)
# Cell 1
!pip install transformers datasets torch accelerate peft -q

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

import json
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from pathlib import Path
from peft import LoraConfig, get_peft_model, TaskType

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

DATA_DIR = BASE_DIR / "data/rlhf/preference_data"
MODEL_DIR = BASE_DIR / "models/sft_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

Using device: cuda


In [ ]:

# Load preference data for SFT
with open(DATA_DIR / 'train_preferences.json', 'r') as f:
    train_data = json.load(f)

with open(DATA_DIR / 'val_preferences.json', 'r') as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

Train: 400, Val: 100


In [ ]:

# Prepare SFT dataset (only chosen responses)
def prepare_sft_data(data):
    sft_examples = []
    for item in data:
        text = f"### Question: {item['prompt']}\n### Answer: {item['chosen']}"
        sft_examples.append({'text': text})
    return sft_examples

train_sft = prepare_sft_data(train_data)
val_sft = prepare_sft_data(val_data)

train_dataset = Dataset.from_list(train_sft)
val_dataset = Dataset.from_list(val_sft)

print(f"SFT Train: {len(train_dataset)}, Val: {len(val_dataset)}")

SFT Train: 400, Val: 100


In [ ]:

# Load base model with LoRA
MODEL_NAME = "gpt2-medium"  # 355M parameters

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

print(f"Base model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

Base model loaded: gpt2-medium
Parameters: 354,823,168


In [ ]:

# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,325,376 || all params: 359,148,544 || trainable%: 1.2043


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print("Datasets tokenized")

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Datasets tokenized


In [ ]:

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",  # Changed from evaluation_strategy
    eval_steps=200,
    save_steps=200,
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to="none"
)

In [ ]:

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Trainer initialized")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer initialized


In [ ]:

# Train
print("\nStarting SFT training...")
trainer.train()



Starting SFT training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


TrainOutput(global_step=75, training_loss=3.980092264811198, metrics={'train_runtime': 42.539, 'train_samples_per_second': 28.209, 'train_steps_per_second': 1.763, 'total_flos': 263646320640000.0, 'train_loss': 3.980092264811198, 'epoch': 3.0})

In [ ]:

# Save model
trainer.save_model(MODEL_DIR / "final")
tokenizer.save_pretrained(MODEL_DIR / "final")

print(f"\nModel saved to {MODEL_DIR / 'final'}")


Model saved to /content/drive/MyDrive/multi_agent_rag_project/models/sft_model/final


In [ ]:

# Test generation
model.eval()
test_prompt = "### Question: What are transformers in deep learning?\n### Answer:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=200,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nTest generation:")
print(generated_text)

print("\nNotebook 08 Complete!")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Test generation:
### Question: What are transformers in deep learning?
### Answer: Based on the research paper 'Computing the Deep Learning Model Based on Numerical Models', we implemented a deep learning model that operates in a multivariate linear space. We demonstrate the key findings from the study. Based on the research paper 'Computing the Deep Learning Model Based on Numerical Models', we implemented a deep learning model that operates in a multivariate linear space. We demonstrate the key findings from the study. Based on the research paper 'Computing the Deep Learning Model Based on Numerical Models', we implemented a deep learning model that operates in a multivariate linear space. We demonstrate the key findings from the study. Based on the research paper 'Computing the Deep Learning Model Based on Numerical Models', we implemented a deep learning model that operates in a multivariate linear space. We demonstrate the key findings from the study. Based on the research study 